# Grazioso Salvare — CS 340 Project Two
Jordon Osborne

**Before submission:** Confirm rescue profiles with the Dashboard Specifications Document; add the original CRUD module and official logo; test in Codio.

In [1]:
# CS 340 Project Two — Grazioso Salvare | Jordon Osborne
# Requires CRUD_Python_Module.py, MongoDB in Codio, and the provided logo.
import os, base64
import pandas as pd
import plotly.express as px
import dash_leaflet as dl
from jupyter_dash import JupyterDash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
from CRUD_Python_Module import AnimalShelter

shelter = AnimalShelter()

# Rescue profiles follow page 4 of the official CS 340 Dashboard Specifications Document.
BREEDS = {
    'Water Rescue': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland'],
    'Mountain or Wilderness Rescue': ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog', 'Siberian Husky', 'Rottweiler'],
    'Disaster or Individual Tracking': ['Doberman Pinscher', 'German Shepherd', 'Golden Retriever', 'Bloodhound', 'Rottweiler']
}

def rescue_query(rescue_type):
    if rescue_type == 'Water Rescue':
        return {'animal_type': 'Dog', 'breed': {'$regex': '^(?:' + '|'.join(__import__('re').escape(b) for b in BREEDS[rescue_type]) + ')(?: Mix)?$', '$options': 'i'},
                'sex_upon_outcome': 'Intact Female', 'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}}
    if rescue_type == 'Mountain or Wilderness Rescue':
        return {'animal_type': 'Dog', 'breed': {'$regex': '^(?:' + '|'.join(__import__('re').escape(b) for b in BREEDS[rescue_type]) + ')(?: Mix)?$', '$options': 'i'},
                'sex_upon_outcome': 'Intact Male', 'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}}
    if rescue_type == 'Disaster or Individual Tracking':
        return {'animal_type': 'Dog', 'breed': {'$regex': '^(?:' + '|'.join(__import__('re').escape(b) for b in BREEDS[rescue_type]) + ')(?: Mix)?$', '$options': 'i'},
                'sex_upon_outcome': 'Intact Male', 'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}}
    return {}

def fetch_animals(filter_type):
    records = shelter.read(rescue_query(filter_type))
    frame = pd.DataFrame.from_records(records)
    return frame.drop(columns=['_id', 'Unnamed: 0'], errors='ignore')

initial = fetch_animals('Reset')
columns = [{'name': c.replace('_', ' ').title(), 'id': c} for c in initial.columns]

# Use the supplied high-resolution Grazioso Salvare logo.
os.chdir('/home/codio/workspace/code_files')
logo_candidates = ['Grazioso_Salvare_Logo.png', 'GraziosoSalvareLogo.png', 'grazioso_salvare_logo.png']
logo_file = next((x for x in logo_candidates if os.path.isfile(x)), None)
logo = html.Img(src='data:image/png;base64,' + base64.b64encode(open(logo_file, 'rb').read()).decode(),
                style={'height': '85px'}) if logo_file else html.Div('Grazioso Salvare', style={'fontSize':'25px','fontWeight':'bold'})

app = JupyterDash(__name__)
app.layout = html.Div([
    html.Center([html.A(logo, href='https://www.snhu.edu', target='_blank'), html.H2('Animal Rescue Candidate Dashboard'),
                 html.P('Developed by Jordon Osborne | CS 340 Project Two')]),
    html.Hr(),
    html.Label('Select rescue training category:'),
    dcc.RadioItems(id='filter-type', options=[{'label':x,'value':x} for x in
       ['Reset','Water Rescue','Mountain or Wilderness Rescue','Disaster or Individual Tracking']],
       value='Reset', labelStyle={'display':'inline-block','marginRight':'20px'}),
    html.P(id='results-count'),
    dash_table.DataTable(id='datatable-id', columns=columns, data=initial.to_dict('records'),
        page_size=10, page_action='native', sort_action='native', filter_action='native',
        row_selectable='single', selected_rows=[], style_table={'overflowX':'auto'},
        style_cell={'textAlign':'left','minWidth':'110px','maxWidth':'220px','overflow':'hidden','textOverflow':'ellipsis'},
        style_header={'fontWeight':'bold','backgroundColor':'#e8f0f1'},
        style_data_conditional=[{'if':{'state':'selected'},'backgroundColor':'#cbe8e9'}]),
    html.Div([html.Div(id='graph-id',style={'flex':'1','minWidth':'360px'}),
              html.Div(id='map-id',style={'flex':'1','minWidth':'360px'})],
             style={'display':'flex','flexWrap':'wrap','gap':'20px','marginTop':'20px'})
], style={'fontFamily':'Arial, sans-serif','padding':'22px'})

@app.callback([Output('datatable-id','data'),Output('datatable-id','selected_rows'),Output('results-count','children')],
              [Input('filter-type','value')])
def update_dashboard(filter_type):
    frame = fetch_animals(filter_type)
    return frame.to_dict('records'), [], f'{len(frame):,} matching animals — {filter_type}'

@app.callback(Output('graph-id','children'),[Input('datatable-id','derived_virtual_data'),Input('datatable-id','data')])
def update_graphs(view_data, table_data):
    rows = view_data if view_data is not None else table_data
    frame = pd.DataFrame(rows or [])
    if frame.empty or 'breed' not in frame:
        return html.P('No animals match the selected criteria.')
    counts = frame['breed'].fillna('Unknown').value_counts().head(10).reset_index()
    counts.columns = ['Breed','Count']
    fig = px.bar(counts, x='Count', y='Breed', orientation='h', title='Top 10 Breeds in Current Table')
    fig.update_layout(yaxis={'categoryorder':'total ascending'}, margin={'l':15,'r':15,'t':55,'b':20})
    return dcc.Graph(figure=fig, config={'displayModeBar':False})

@app.callback(Output('map-id','children'),
              [Input('datatable-id','derived_virtual_data'),Input('datatable-id','data'),
               Input('datatable-id','derived_virtual_selected_rows')])
def update_map(view_data, table_data, selected_rows):
    rows = view_data if view_data is not None else table_data
    frame = pd.DataFrame(rows or [])
    if frame.empty or not {'location_lat','location_long'}.issubset(frame.columns):
        return html.P('No map coordinates are available for this selection.')
    frame['location_lat'] = pd.to_numeric(frame['location_lat'], errors='coerce')
    frame['location_long'] = pd.to_numeric(frame['location_long'], errors='coerce')
    frame = frame.dropna(subset=['location_lat','location_long']).reset_index(drop=True)
    if frame.empty:
        return html.P('No valid map coordinates are available.')
    index = selected_rows[0] if selected_rows and selected_rows[0] < len(frame) else 0
    animal = frame.iloc[index]
    point = [float(animal['location_lat']),float(animal['location_long'])]
    return dl.Map(center=point, zoom=10, style={'height':'420px','width':'100%'},children=[
        dl.TileLayer(), dl.Marker(position=point,children=[
            dl.Tooltip(str(animal.get('breed','Animal'))),
            dl.Popup(html.Div([html.B(str(animal.get('name','Unnamed animal'))),
                               html.P(str(animal.get('breed','')))]))])])

# Run in Codio/Jupyter. Select each rescue filter and capture screenshots for the README.
app.run_server(
    host="0.0.0.0",
    port=8050,
    mode="external",
    debug=False
)


 * Running on all addresses.
 * Running on http://10.11.201.179:8050/ (Press CTRL+C to quit)
127.0.0.1 - - [15/Oct/2026 14:22:59] "GET /_alive_d65edbd6-d49e-417f-9e68-40b93ad74c5c HTTP/1.1" 200 -


Dash app running on http://0.0.0.0:8050/


In [2]:
from IPython.display import IFrame, display
display(IFrame(src="http://127.0.0.1:8050/", width="100%", height=700))

In [3]:
import os

print("Current folder:", os.getcwd())
print("Logo exists:", os.path.isfile("Grazioso_Salvare_Logo.png"))
print("Logo files:", [f for f in os.listdir() if "Grazioso" in f])

Current folder: /home/codio/workspace/code_files
Logo exists: True
Logo files: ['Grazioso_Salvare_Logo.png']
